# Lab 4: Cleaning II (Text, Dates, Encodings)

**DSA 405 · Week 4**

| | |
|---|---|
| **In class** | Friday, Sep 11 |
| **A4 due** | Thursday, Sep 17, 11:59 PM |
| **File** | `wolfpack_dining_raw.csv` |
| **Also this week** | **Bench Check 1** sign-up opens (slots run Weeks 5–7, in class) |
| **Time** | ~25 min in class, ~55 min at home |

## Overview

Cleaning text is the slowest part of most wrangling work, and this week we take a look at
three common causes: strings that almost match, dates that arrive in three formats,
and text that went through the wrong encoding somewhere and now shows `Ã©` where an `é`
should be. None of it is hard once you've seen the pattern, but each one is slow the
first time you encounter it, and that's normal.

In [1]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, io

DATA = "https://raw.githubusercontent.com/jon-holt/DSA-405-Student/main/datasets/"
# DATA = "data/raw/"          # local users


def load(filename, kind="csv", **kw):
    """Read a class file whether DATA is a URL or a local folder."""
    path = DATA + filename
    if kind == "csv":
        return pd.read_csv(path, **kw)
    if kind == "excel":
        return pd.read_excel(path, **kw)
    if kind == "text":
        return requests.get(path, timeout=30).text if path.startswith("http") else open(path).read()
    if kind == "json":
        if path.startswith("http"):
            return requests.get(path, timeout=30).json()
        import json as _j
        return _j.load(open(path))
    raise ValueError(kind)


pd.set_option("display.width", 160)
print("pandas", pd.__version__)

pandas 2.2.3


---
# Part 1: Explore (in class)

## Task 1.1: 27 strings, 6 categories

In Lab 3 we handed you a category canonicalizer as a given. This week you build your
own, and the place to start is reading the raw values:

In [2]:
dining = load("wolfpack_dining_raw.csv", dtype=str)

for v in sorted(dining.category.dropna().unique()):
    print(repr(v))

' Coffee'
'C-Store'
'CAFE'
'COFFEE'
'Cafe'
'Café'
'Coffee'
'Coffee '
'Coffee Shop'
'Convenience'
'Convenience Store'
'DINING HALL'
'Dining Hall'
'Dining hall'
'Fast Casual'
'Fast-Casual'
'FastCasual'
'Food Truck'
'Food truck'
'FoodTruck'
'cafe'
'coffee'
'convenience'
'dining hall'
'fast casual'
'food truck'
'foodtruck'


Take your time with the `repr()` output. The quotes make leading and trailing
whitespace visible, which is exactly what `print` would hide. As you can see, there are many issues we need to address: case (`COFFEE`/`coffee`), padding (`' Coffee'`, `'Coffee '`), punctuation
(`Fast-Casual`/`FastCasual`), synonyms (`C-Store`, `Coffee Shop`), and one Unicode
variant (`Café`).

Normalize first and map second. Most of the variants collapse under mechanical
normalization, and the few true synonyms left over need a hand-written map, which by
then is short enough to verify by eye:

In [3]:
# Normalize and clean the 'category' column in the dining DataFrame
norm = (dining.category
        .str.strip()
        .str.lower()
        .str.replace("-", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True))

print("after normalizing :", norm.nunique(), "distinct")

# Define a dictionary of synonyms to standardize category names
SYNONYMS = {"c store": "convenience", "convenience store": "convenience",
            "coffee shop": "coffee", "café": "cafe",
            "fastcasual": "fast casual", "foodtruck": "food truck"}
dining["category_clean"] = norm.replace(SYNONYMS)

print("after mapping     :", dining.category_clean.nunique(), "distinct")
print(dining.category_clean.value_counts().to_dict())

unmapped = dining.category_clean.value_counts()
assert dining.category_clean.nunique() == 6, "still fragmented — check the map"
print("6 categories. The assert will catch it if a new variant ever sneaks in.")

after normalizing : 12 distinct
after mapping     : 6 distinct
{'food truck': 107, 'coffee': 57, 'fast casual': 53, 'cafe': 51, 'dining hall': 50, 'convenience': 48}
6 categories. The assert will catch it if a new variant ever sneaks in.


## Task 1.2: Dates in three formats

Look at some values in `inspection_date`:

In [4]:
print(dining.inspection_date.dropna().sample(8, random_state=405).tolist())

['July 25, 2024', '30-Dec-2024', 'September 9, 2025', 'September 17, 2024', '2024-01-25', 'January 26, 2024', '6-Feb-2024', '02/24/2025']


`03/20/2025`, `2024-07-18`, `January 4, 2025`, some with timestamps appended. Mixed
formats in one column are the classic signature of data merged from two systems.

`format="mixed"` lets each value be parsed by its own format. That convenience is worth
verifying afterward, because a date parser reports success even when it guessed wrong:

In [5]:
dates = pd.to_datetime(dining.inspection_date, format="mixed")

print("failed to parse:", dates.isna().sum())
print("range:", dates.min().date(), "to", dates.max().date())

assert dates.notna().all(), "some dates failed to parse"
assert dates.dt.year.isin([2024, 2025]).all(), "a year outside 2024-2025 appeared"
print("all dates parsed, all years in range")

failed to parse: 0
range: 2024-01-06 to 2025-12-30
all dates parsed, all years in range


Zero failures and a plausible range. Notice that we asserted the years rather than
eyeballing them. A date like `04/07/2025` parses as April 7 and as July 4, and one of
those is silently wrong, so an assertion on the outcome (range, year, weekday patterns)
is what catches a parser that guessed a different format than you intended.

## Task 1.3: Mojibake, the é that became Ã©

Print a few location names with `repr()`:

In [6]:
damaged = dining[dining.location_name.str.contains("Ã", na=False)]
print(f"rows with Ã in location_name: {len(damaged)} of {len(dining)}")
print()
for v in damaged.location_name.head(4):
    print(repr(v))

rows with Ã in location_name: 62 of 366

'CrÃ¨me Scott Pizza Window'
' BÃ¡nh Fountain Market'
'BÃ¡nh Polk Snack Shop'
'JalapeÃ±o Withers Juice Bar '


62 rows. `CafÃ©` is `Café` that was written as UTF-8 and read as Latin-1, so each
2-byte character split into two 1-byte ones. The pattern has a name, mojibake, and two
properties worth knowing. It is diagnosable, because the `Ã` is a fingerprint, and it
is repairable, because the information is still present, only mis-decoded. You'll come
back to these 62 rows in Task 2.4, where you’ll be asked to think about the consequences of these mis-coded characters.

---
## Checkpoint: submit before leaving class

1. How many distinct category strings did mechanical normalization alone resolve, and how
   many needed the synonym map?
2. Which two assertions pinned the parsed dates in Task 1.2, and what failure would each
   catch?
3. How many rows carry mojibake in `location_name`, and is the damage recoverable?

*Answers here.*
1. Mechanical normalization resolved 15 categories and reduced the 27 orginial to 12 distinct categories. Synonym mapping was need for 6 more of the categories.
2. The two assertions were:
assert dates.notna().all() which checks that every date was successfully parsed and catches any dates that become missing (NaT).
assert dates.dt.year.isin([2024, 2025]).all() checks that all parsed dates fall within the expected years of 2024–2025, catching dates that may have been parsed incorrectly or fall outside the expected range.
3. 62 rows out of 366 contain mojibake in location_name. The damage is recoverable because the original character information is still present; it was just decoded using the wrong character encoding.

---
# Part 2: A4 (Text, Regex & Dates)

Four tasks. The first three are practice with today's tools. Task 2.4 is where to
spend your time, because it asks you to connect a technical choice to its consequence,
and that connection is the skill this week is actually about.

## Task 2.1: Location names, normalized

`location_name` has the same layered damage as `category`: case, padding, internal
whitespace. Repair `location_name` by following the same steps we used to repair `category`: Normalize it (leave the mojibake in place; Task 2.2 handles it), report
distinct counts before and after, and show three name variants that collapsed into one.

In [7]:
# your normalization

## Task 2.2: Extraction with patterns

Three small regex jobs on this file:

**a)** `avg_ticket` mixes `$24.87` with sentinels (`unknown`, `-999`, whitespace).
How many values in this column have a standard currency pattern? What
are the values that do not?

**b)** Extract the number from every currency value by using this: (`.str.extract(r"([0-9.]+)")`,
then `to_numeric`) and state how many real prices we now have.

**c)** Count the rows whose `location_name` contains the mojibake fingerprint `Ã`.

In [8]:
# your three patterns

## Task 2.3: Dates, with assumptions

Parse `inspection_date` like we did in class, then add at least three assertions: parse
completeness, year range, and one more (a good third: no date in the future). For each
assertion, add one clause naming the wrong parse it would catch (in other words, what's an example of a problem that your parse statement would catch).

In [9]:
# your parse and assertions

## Task 2.4: The consequence of an encoding decision

A teammate proposes removing all rows that contain mojibake. Before agreeing or
objecting, measure what the drop would do.

1. Count rows per `campus_zone` before any drop. State which zones are smallest, and by
   how much.
2. Drop every row with mojibake in `location_name`, and count again.
3. Report the zone that changed most, and include specific numbers to back up your statement (in other words, how much did the zone change when you dropped the rows).

Then write a paragraph about your findings. As you have seen, a choice about
text encoding has produced a false claim about where the university has dining
locations. Name the zone that changed the most (from part 3 above) and describe what the consequences were of dropping those rows. Finish
with what should happen to the mojibake rows instead of dropping them.

In [10]:
# your before/after counts

*Paragraph here.*

---
## AI use note

Tell me which AI tools you used here and what you used them for, in a sentence or two.
If you didn't use any, write "none."

*Answer here.*

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. `File > Download > Download .ipynb`
3. Rename to `DSA405_002_FA26_A4_[yourUnityID].ipynb`
4. Upload to the **A4** space on Moodle.

The **Checkpoint** section goes separately to **Week 4 In-Class Activity** before the
end of class on Friday. A4 is due **Thursday, Sep 17, 11:59 PM**.